In [71]:
import pandas as pd
df = pd.read_csv(r"C:\Users\LAPTOP WORLD\Desktop\Datasets\social_media_performance.csv")
print(df.head())


   post_id  platform content_type       topic language region  \
0        1  LinkedIn      article  Technology       UR     BR   
1        2  LinkedIn         poll      Health       FR     JP   
2        3  LinkedIn      article      Travel       HI     FR   
3        4  LinkedIn        image      Sports       DE     DE   
4        5  LinkedIn         poll    Business       DE     US   

         post_datetime                                           hashtags  \
0  2025-04-25 09:47:00  #AI #Innovation #TechTrends #Programming #Codi...   
1  2025-10-29 09:44:00  #Fitness #Nutrition #Wellness #Health #MentalH...   
2  2025-02-10 14:12:00  #Travel #Journey #Adventure #Tourism #ExploreM...   
3  2025-04-18 22:41:00                         #Cricket #Workout #Fitness   
4  2025-04-28 10:17:00             #Entrepreneur #Leadership #StartupLife   

   sentiment_score  views  likes  comments  shares  engagement_rate  is_viral  
0             0.76  37781   1202       462     185           0.049

In [16]:
df.groupby("sentiment_score")["views"].mean().sort_values(ascending = False)

#best hour

df.dropna(inplace = True)
df["post_datetime"] = pd.to_datetime(df["post_datetime"])
df["hour"]= df["post_datetime"].dt.hour
df.groupby("hour")["engagement_rate"].mean().sort_values(ascending = False)

hour
18    0.144195
19    0.122587
20    0.121416
16    0.118649
6     0.118314
9     0.116094
13    0.115481
1     0.115125
12    0.114983
5     0.113684
10    0.113607
0     0.113511
7     0.111553
8     0.110711
17    0.108923
23    0.108608
11    0.108196
2     0.106913
3     0.104119
4     0.103386
21    0.097489
22    0.096557
15    0.089949
14    0.088577
Name: engagement_rate, dtype: float64

In [19]:
df["total_engagement"] = df["likes"] + df["comments"] + df["shares"]
df["dayofweek"] = df["post_datetime"].dt.dayofweek


In [37]:
df_ml = pd.get_dummies(df, drop_first = True)
x = df_ml.drop(["views", "is_viral"], axis = 1 )


In [38]:
y_views = df_ml["views"]
y_viral = df_ml["is_viral"]


In [74]:
from sklearn.model_selection import train_test_split
x_train, x_test, yv_train, yv_test = train_test_split( x,y_views, test_size = 0.2 , random_state= 42 )
x_train, x_test, yv_train, yv_test = train_test_split(x,y_viral, test_size = 0.2 , random_state = 42 )

In [76]:
from sklearn.ensemble import RandomForestRegressor

model_views = RandomForestRegressor(
    n_estimators = 200,
    max_depth = 10,
    random_state = 42
)
model_views.fit(x_train, yv_train)

pred_views = model_views.predict(x_test)

In [47]:
x_train.dtypes

post_id                                                                                 int64
post_datetime                                                                  datetime64[us]
sentiment_score                                                                       float64
likes                                                                                   int64
comments                                                                                int64
                                                                                    ...      
hashtags_#Workout #Training #Football #Sports #Basketball                                bool
hashtags_#Workout #Training #Sports #Basketball #Fitness #Cricket #Football              bool
hashtags_#Workout #Training #Sports #Basketball #Football #Cricket #Fitness              bool
hashtags_#Workout #Training #Sports #Fitness #Football #Cricket #Basketball              bool
hashtags_#Workout #Training #Sports #Football #Fitness      

In [63]:
df.dtypes

sentiment_score                                                                float64
likes                                                                            int64
comments                                                                         int64
shares                                                                           int64
engagement_rate                                                                float64
                                                                                ...   
hashtags_#Workout #Training #Football #Sports #Basketball                         bool
hashtags_#Workout #Training #Sports #Basketball #Fitness #Cricket #Football       bool
hashtags_#Workout #Training #Sports #Basketball #Football #Cricket #Fitness       bool
hashtags_#Workout #Training #Sports #Fitness #Football #Cricket #Basketball       bool
hashtags_#Workout #Training #Sports #Football #Fitness                            bool
Length: 7989, dtype: object

In [73]:
x = df.drop(columns=['views','post_datetime', 'post_id' , 'plateform',], errors = 'ignore')
x = pd.get_dummies(x, drop_first = True)
y = df['views']
x_train , x_test, yv_train, yv_test = train_test_split(x , y , test_size = 0.2, random_state = 42 )

In [77]:
from sklearn.metrics import r2_score
print("Views R2:" , r2_score(yv_test, pred_views))

Views R2: 1.0


In [82]:
from sklearn.ensemble import RandomForestClassifier
model_viral = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 10,
    random_state = 42
)

model_viral.fit(x_train , yv_train)
pred_viral = model_viral.predict(x_test)

In [83]:
from sklearn.metrics import accuracy_score
print("Viral Accuracy:", accuracy_score(yv_test, pred_viral))

Viral Accuracy: 0.937


In [86]:
# feature importance

import pandas as pd

importance = pd.DataFrame({
    "Feature": x.columns,
    "importance" : model_viral.feature_importances_
}).sort_values(by="importance", ascending = False)

importance.head(10)

,Feature,importance
5,is_viral,0.117385
4,engagement_rate,0.094170
2,comments,0.080077
1,likes,0.075850
3,shares,0.064722
0,sentiment_score,0.031376
15,content_type_video,0.027421
6,platform_LinkedIn,0.024961
8,platform_YouTube,0.021549
12,content_type_poll,0.011410


In [87]:
# Final insights
#1 likes and comments are strongest predictors of virality.
#2 poll type content viral aour views achche la skte hai.